<a href="https://colab.research.google.com/github/DDricko/Ciencia_de_dados_Ebac/blob/main/Mod13/Mod13_Tarefa02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda II

Vamos continuar trabalhando com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('previsao_de_renda.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
2. Rode uma regularização *ridge* com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o $R^2$ na base de testes. Qual o melhor modelo?
3. Faça o mesmo que no passo 2, com uma regressão *LASSO*. Qual método chega a um melhor resultado?
4. Rode um modelo *stepwise*. Avalie o $R^2$ na vase de testes. Qual o melhor resultado?
5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?
6. Partindo dos modelos que você ajustou, tente melhorar o $R^2$ na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.
7. Ajuste uma árvore de regressão e veja se consegue um $R^2$ melhor com ela.

In [24]:
# Importar bibliotecas necessárias
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor
import warnings
warnings.filterwarnings('ignore')

## 1. Preparação dos dados

In [25]:
# Preparar dados para machine learning
df['log_renda'] = np.log(df['renda'])

# Criar variáveis dummy
df_encoded = pd.get_dummies(df, columns=['sexo', 'posse_de_veiculo', 'posse_de_imovel',
                                       'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia'],
                           drop_first=True, dtype=int)

# Definir variáveis X e y
X = df_encoded.drop(['renda', 'log_renda', 'data_ref', 'id_cliente'], axis=1)
y = df_encoded['log_renda']

print(f"Shape original dos dados: {df.shape}")
print(f"Shape após encoding: {X.shape}")
print(f"Variáveis incluídas: {X.columns.tolist()[:10]}...") # mostra só as 10 primeiras

# Dividir em treino e teste (75% / 25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"\nDivisão dos dados:")
print(f"Treino: {X_train.shape[0]} observações")
print(f"Teste: {X_test.shape[0]} observações")

Shape original dos dados: (15000, 16)
Shape após encoding: (15000, 25)
Variáveis incluídas: ['Unnamed: 0', 'qtd_filhos', 'idade', 'tempo_emprego', 'qt_pessoas_residencia', 'sexo_M', 'posse_de_veiculo_True', 'posse_de_imovel_True', 'tipo_renda_Bolsista', 'tipo_renda_Empresário']...

Divisão dos dados:
Treino: 11250 observações
Teste: 3750 observações


In [26]:
# Padronizar as variáveis (importante para regularização)

# Tratar NaNs na coluna 'tempo_emprego' antes da padronização
# Imputar com a média do conjunto de treino para evitar data leakage
mean_tempo_emprego_train = X_train['tempo_emprego'].mean()
X_train['tempo_emprego'] = X_train['tempo_emprego'].fillna(mean_tempo_emprego_train)
X_test['tempo_emprego'] = X_test['tempo_emprego'].fillna(mean_tempo_emprego_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Dados padronizados para regularização")
print(f"Média das variáveis após padronização (treino): {X_train_scaled.mean():.4f}")
print(f"Desvio padrão das variáveis após padronização (treino): {X_train_scaled.std():.4f}")

✅ Dados padronizados para regularização
Média das variáveis após padronização (treino): 0.0000
Desvio padrão das variáveis após padronização (treino): 1.0000


## 2. Regularização Ridge

In [27]:
# Regularização Ridge com diferentes alphas
alphas_ridge = [0, 0.001, 0.005, 0.01, 0.05, 0.1]
resultados_ridge = []

print("=== RESULTADOS RIDGE ===")
for alpha in alphas_ridge:
    # Treinar modelo
    ridge = Ridge(alpha=alpha, random_state=42)
    ridge.fit(X_train_scaled, y_train)

    # Predições
    y_pred_train = ridge.predict(X_train_scaled)
    y_pred_test = ridge.predict(X_test_scaled)

    # Calcular R²
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    resultados_ridge.append({
        'alpha': alpha,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'modelo': ridge
    })

    print(f"Alpha {alpha:>6}: R² treino = {r2_train:.4f} | R² teste = {r2_test:.4f}")

# Melhor modelo Ridge
melhor_ridge = max(resultados_ridge, key=lambda x: x['r2_test'])
print(f"\n🏆 Melhor Ridge: Alpha = {melhor_ridge['alpha']} com R² teste = {melhor_ridge['r2_test']:.4f}")

=== RESULTADOS RIDGE ===
Alpha      0: R² treino = 0.3470 | R² teste = 0.3509
Alpha  0.001: R² treino = 0.3470 | R² teste = 0.3509
Alpha  0.005: R² treino = 0.3470 | R² teste = 0.3509
Alpha   0.01: R² treino = 0.3470 | R² teste = 0.3509
Alpha   0.05: R² treino = 0.3470 | R² teste = 0.3509
Alpha    0.1: R² treino = 0.3470 | R² teste = 0.3509

🏆 Melhor Ridge: Alpha = 0 com R² teste = 0.3509


## 3. Regularização LASSO

In [28]:
# Regularização LASSO com os mesmos alphas
alphas_lasso = [0.001, 0.005, 0.01, 0.05, 0.1]  # alpha=0 não funciona no LASSO
resultados_lasso = []

print("=== RESULTADOS LASSO ===")
for alpha in alphas_lasso:
    # Treinar modelo
    lasso = Lasso(alpha=alpha, random_state=42, max_iter=2000)
    lasso.fit(X_train_scaled, y_train)

    # Predições
    y_pred_train = lasso.predict(X_train_scaled)
    y_pred_test = lasso.predict(X_test_scaled)

    # Calcular R²
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Contar variáveis selecionadas (coef != 0)
    n_vars = sum(lasso.coef_ != 0)

    resultados_lasso.append({
        'alpha': alpha,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'n_variaveis': n_vars,
        'modelo': lasso
    })

    print(f"Alpha {alpha:>6}: R² treino = {r2_train:.4f} | R² teste = {r2_test:.4f} | {n_vars} variáveis")

# Melhor modelo LASSO
melhor_lasso = max(resultados_lasso, key=lambda x: x['r2_test'])
print(f"\n🏆 Melhor LASSO: Alpha = {melhor_lasso['alpha']} com R² teste = {melhor_lasso['r2_test']:.4f}")
print(f"   {melhor_lasso['n_variaveis']} variáveis selecionadas")

=== RESULTADOS LASSO ===
Alpha  0.001: R² treino = 0.3469 | R² teste = 0.3506 | 23 variáveis
Alpha  0.005: R² treino = 0.3460 | R² teste = 0.3504 | 20 variáveis
Alpha   0.01: R² treino = 0.3443 | R² teste = 0.3491 | 14 variáveis
Alpha   0.05: R² treino = 0.3259 | R² teste = 0.3304 | 5 variáveis
Alpha    0.1: R² treino = 0.2966 | R² teste = 0.3004 | 2 variáveis

🏆 Melhor LASSO: Alpha = 0.001 com R² teste = 0.3506
   23 variáveis selecionadas


## 4. Modelo Stepwise (simplificado)

In [30]:
# Usar LASSO como proxy para stepwise (já faz seleção de variáveis)
from sklearn.linear_model import LinearRegression

# Modelo stepwise: usar as variáveis selecionadas pelo melhor LASSO
lasso_melhor = melhor_lasso['modelo']
vars_selecionadas = X.columns[lasso_melhor.coef_ != 0]

print(f"=== MODELO STEPWISE (baseado no LASSO) ===")
print(f"Variáveis selecionadas ({len(vars_selecionadas)}):")
for var in vars_selecionadas:
    print(f"  - {var}")

# Treinar modelo linear apenas com variáveis selecionadas
X_train_selected = X_train[vars_selecionadas]
X_test_selected = X_test[vars_selecionadas]


if 'tempo_emprego' in vars_selecionadas:
    mean_tempo_emprego_train = X_train_selected['tempo_emprego'].mean()
    X_train_selected = X_train_selected.fillna({'tempo_emprego': mean_tempo_emprego_train})
    X_test_selected = X_test_selected.fillna({'tempo_emprego': mean_tempo_emprego_train})

stepwise = LinearRegression()
stepwise.fit(X_train_selected, y_train)

# Predições e R²
y_pred_train_stepwise = stepwise.predict(X_train_selected)
y_pred_test_stepwise = stepwise.predict(X_test_selected)

r2_train_stepwise = r2_score(y_train, y_pred_train_stepwise)
r2_test_stepwise = r2_score(y_test, y_pred_test_stepwise)

print(f"\nResultados Stepwise:")
print(f"R² treino: {r2_train_stepwise:.4f}")
print(f"R² teste: {r2_test_stepwise:.4f}")

=== MODELO STEPWISE (baseado no LASSO) ===
Variáveis selecionadas (23):
  - Unnamed: 0
  - idade
  - tempo_emprego
  - qt_pessoas_residencia
  - sexo_M
  - posse_de_veiculo_True
  - posse_de_imovel_True
  - tipo_renda_Bolsista
  - tipo_renda_Empresário
  - tipo_renda_Pensionista
  - tipo_renda_Servidor público
  - educacao_Pós graduação
  - educacao_Secundário
  - educacao_Superior completo
  - educacao_Superior incompleto
  - estado_civil_Separado
  - estado_civil_Solteiro
  - estado_civil_União
  - estado_civil_Viúvo
  - tipo_residencia_Com os pais
  - tipo_residencia_Comunitário
  - tipo_residencia_Estúdio
  - tipo_residencia_Governamental

Resultados Stepwise:
R² treino: 0.3469
R² teste: 0.3505


## 5. Comparação de todos os modelos

In [32]:
# Criar tabela comparativa de todos os modelos
import pandas as pd

comparacao_modelos = []

# Ridge
for resultado in resultados_ridge:
    comparacao_modelos.append({
        'Modelo': f"Ridge (α={resultado['alpha']})",
        'R²_Treino': resultado['r2_train'],
        'R²_Teste': resultado['r2_test'],
        'N_Variáveis': len(X.columns),
        'Tipo': 'Ridge'
    })

# LASSO
for resultado in resultados_lasso:
    comparacao_modelos.append({
        'Modelo': f"LASSO (α={resultado['alpha']})",
        'R²_Treino': resultado['r2_train'],
        'R²_Teste': resultado['r2_test'],
        'N_Variáveis': resultado['n_variaveis'],
        'Tipo': 'LASSO'
    })

# Stepwise
comparacao_modelos.append({
    'Modelo': 'Stepwise',
    'R²_Treino': r2_train_stepwise,
    'R²_Teste': r2_test_stepwise,
    'N_Variáveis': len(vars_selecionadas),
    'Tipo': 'Stepwise'
})

# Criar DataFrame
df_comparacao = pd.DataFrame(comparacao_modelos)
df_comparacao = df_comparacao.sort_values('R²_Teste', ascending=False)

print("=== COMPARAÇÃO COMPLETA DOS MODELOS ===")
print(df_comparacao.round(4))

# Identificar melhor modelo
melhor_modelo = df_comparacao.iloc[0]
print(f"\n🏆 MELHOR MODELO: {melhor_modelo['Modelo']}")
print(f"   R² Teste: {melhor_modelo['R²_Teste']:.4f}")
print(f"   Variáveis: {melhor_modelo['N_Variáveis']}")

# Comparar Ridge vs LASSO
melhor_ridge_r2 = max(resultado['r2_test'] for resultado in resultados_ridge)
melhor_lasso_r2 = max(resultado['r2_test'] for resultado in resultados_lasso)

print(f"\n🔍 Ridge vs LASSO:")
print(f"   Melhor Ridge R²: {melhor_ridge_r2:.4f}")
print(f"   Melhor LASSO R²: {melhor_lasso_r2:.4f}")
if melhor_lasso_r2 > melhor_ridge_r2:
    print("   ✅ LASSO tem melhor performance")
else:
    print("   ✅ Ridge tem melhor performance")

=== COMPARAÇÃO COMPLETA DOS MODELOS ===
             Modelo  R²_Treino  R²_Teste  N_Variáveis      Tipo
0       Ridge (α=0)     0.3470    0.3509           25     Ridge
1   Ridge (α=0.001)     0.3470    0.3509           25     Ridge
2   Ridge (α=0.005)     0.3470    0.3509           25     Ridge
3    Ridge (α=0.01)     0.3470    0.3509           25     Ridge
4    Ridge (α=0.05)     0.3470    0.3509           25     Ridge
5     Ridge (α=0.1)     0.3470    0.3509           25     Ridge
6   LASSO (α=0.001)     0.3469    0.3506           23     LASSO
11         Stepwise     0.3469    0.3505           23  Stepwise
7   LASSO (α=0.005)     0.3460    0.3504           20     LASSO
8    LASSO (α=0.01)     0.3443    0.3491           14     LASSO
9    LASSO (α=0.05)     0.3259    0.3304            5     LASSO
10    LASSO (α=0.1)     0.2966    0.3004            2     LASSO

🏆 MELHOR MODELO: Ridge (α=0)
   R² Teste: 0.3509
   Variáveis: 25

🔍 Ridge vs LASSO:
   Melhor Ridge R²: 0.3509
   Melhor LASSO

## 6. Tentativa criativa de melhoria

In [33]:
# Criar variáveis de engenharia/interação
X_enhanced = X.copy()

# 1. Interações importantes
X_enhanced['idade_x_tempo_emprego'] = X['idade'] * X['tempo_emprego']
X_enhanced['idade_quadrada'] = X['idade'] ** 2
X_enhanced['tempo_emprego_quadrado'] = X['tempo_emprego'] ** 2

# 2. Variável composta: experiência relativa
X_enhanced['experiencia_relativa'] = X['tempo_emprego'] / (X['idade'] + 1e-6)  # evitar divisão por zero

print("=== MODELO MELHORADO ===")
print(f"Variáveis originais: {X.shape[1]}")
print(f"Variáveis após engenharia: {X_enhanced.shape[1]}")

# Dividir os novos dados
X_train_enh, X_test_enh, y_train, y_test = train_test_split(X_enhanced, y, test_size=0.25, random_state=42)

if 'tempo_emprego' in X_train_enh.columns:
    mean_tempo_emprego_enh_train = X_train_enh['tempo_emprego'].mean()
    X_train_enh['tempo_emprego'] = X_train_enh['tempo_emprego'].fillna(mean_tempo_emprego_enh_train)
    X_test_enh['tempo_emprego'] = X_test_enh['tempo_emprego'].fillna(mean_tempo_emprego_enh_train)

# As variáveis criadas também podem ter NaNs, preencher com 0 após a imputação da base.
X_train_enh = X_train_enh.fillna(0)
X_test_enh = X_test_enh.fillna(0)

# Padronizar
scaler_enh = StandardScaler()
X_train_enh_scaled = scaler_enh.fit_transform(X_train_enh)
X_test_enh_scaled = scaler_enh.transform(X_test_enh)

# Treinar LASSO melhorado
lasso_melhorado = Lasso(alpha=melhor_lasso['alpha'], random_state=42, max_iter=2000)
lasso_melhorado.fit(X_train_enh_scaled, y_train)

# Avaliar
y_pred_train_enh = lasso_melhorado.predict(X_train_enh_scaled)
y_pred_test_enh = lasso_melhorado.predict(X_test_enh_scaled)

r2_train_enh = r2_score(y_train, y_pred_train_enh)
r2_test_enh = r2_score(y_test, y_pred_test_enh)

print(f"\nResultados do modelo melhorado:")
print(f"R² treino: {r2_train_enh:.4f}")
print(f"R² teste: {r2_test_enh:.4f}")
print(f"Variáveis selecionadas: {sum(lasso_melhorado.coef_ != 0)}")

# Comparar com melhor modelo anterior
melhoria = r2_test_enh - melhor_modelo['R²_Teste']
print(f"\nComparação com melhor modelo anterior:")
print(f"Melhoria: {melhoria:+.4f} em R² teste")
if melhoria > 0:
    print("✅ Modelo melhorou!")
else:
    print("❌ Modelo não melhorou")

=== MODELO MELHORADO ===
Variáveis originais: 25
Variáveis após engenharia: 29

Resultados do modelo melhorado:
R² treino: 0.3494
R² teste: 0.3532
Variáveis selecionadas: 24

Comparação com melhor modelo anterior:
Melhoria: +0.0023 em R² teste
✅ Modelo melhorou!


## 7. Árvore de Regressão

In [34]:
# Árvore de Regressão (não precisa padronização)

# Imputar NaNs em X_train e X_test para 'tempo_emprego' antes de usar na árvore
# A árvore de regressão do sklearn não lida com NaNs por padrão.
mean_tempo_emprego_tree_train = X_train['tempo_emprego'].mean()
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()
X_train_imputed['tempo_emprego'] = X_train_imputed['tempo_emprego'].fillna(mean_tempo_emprego_tree_train)
X_test_imputed['tempo_emprego'] = X_test_imputed['tempo_emprego'].fillna(mean_tempo_emprego_tree_train)

tree = DecisionTreeRegressor(
    max_depth=10,        # Evitar overfitting
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

tree.fit(X_train_imputed, y_train)

# Predições
y_pred_train_tree = tree.predict(X_train_imputed)
y_pred_test_tree = tree.predict(X_test_imputed)

# Métricas
r2_train_tree = r2_score(y_train, y_pred_train_tree)
r2_test_tree = r2_score(y_test, y_pred_test_tree)

print("=== ÁRVORE DE REGRESSÃO ===")
print(f"R² treino: {r2_train_tree:.4f}")
print(f"R² teste: {r2_test_tree:.4f}")
print(f"Profundidade da árvore: {tree.get_depth()}")
print(f"Número de folhas: {tree.get_n_leaves()}")

# Importância das variáveis (top 10)
importancias = pd.DataFrame({
    'Variavel': X_train_imputed.columns, # Usar colunas do X_train_imputed
    'Importancia': tree.feature_importances_
}).sort_values('Importancia', ascending=False)

print(f"\nTop 10 variáveis mais importantes:")
for i, (_, row) in enumerate(importancias.head(10).iterrows()):
    print(f"{i+1:>2}. {row['Variavel']:<25} {row['Importancia']:.4f}")

print(f"\nComparação árvore vs modelos lineares:")
print(f"Melhor modelo linear: R² = {max(r2_test_enh, melhor_modelo['R²_Teste']):.4f}")
print(f"Árvore de regressão:  R² = {r2_test_tree:.4f}")

if r2_test_tree > max(r2_test_enh, melhor_modelo['R²_Teste']):
    print("✅ Árvore superou modelos lineares!")
else:
    print("❌ Modelos lineares ainda são melhores")

=== ÁRVORE DE REGRESSÃO ===
R² treino: 0.4597
R² teste: 0.3312
Profundidade da árvore: 10
Número de folhas: 317

Top 10 variáveis mais importantes:
 1. tempo_emprego             0.5355
 2. sexo_M                    0.3036
 3. idade                     0.0470
 4. Unnamed: 0                0.0344
 5. tipo_renda_Empresário     0.0130
 6. posse_de_imovel_True      0.0118
 7. qt_pessoas_residencia     0.0100
 8. tipo_renda_Servidor público 0.0077
 9. educacao_Superior completo 0.0074
10. posse_de_veiculo_True     0.0069

Comparação árvore vs modelos lineares:
Melhor modelo linear: R² = 0.3532
Árvore de regressão:  R² = 0.3312
❌ Modelos lineares ainda são melhores
